In [8]:
import pandas as pd

df = pd.read_excel("Headspace Reddit Review.xlsx")

df.head()
df.columns


Index(['primary_id', 'parent_id', 'type', 'subreddit', 'url', 'datetime',
       'username', 'text_raw', 'text_clean', 'thematic_notes', 'final_theme',
       'Search Entry'],
      dtype='object')

In [9]:
df.to_csv(
    "Headspace_Reddit_CLEAN.csv",
    index=False,
    encoding="utf-8-sig"
)


In [10]:
df = pd.read_csv("Headspace_Reddit_CLEAN.csv", encoding="utf-8-sig")

df.head()
df.columns


Index(['primary_id', 'parent_id', 'type', 'subreddit', 'url', 'datetime',
       'username', 'text_raw', 'text_clean', 'thematic_notes', 'final_theme',
       'Search Entry'],
      dtype='object')

In [11]:
# Topic clustering (TF–IDF + KMeans) after your cleaned CSV works
# Assumes you already have: Headspace_Reddit_CLEAN.csv

import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# -----------------------------
# 1) Load clean CSV
# -----------------------------
df = pd.read_csv("Headspace_Reddit_CLEAN.csv", encoding="utf-8-sig")

# Use text_raw (since your text_clean was empty originally)
df["text_raw"] = df["text_raw"].fillna("").astype(str)

# -----------------------------
# 2) Create text_clean in Python
# -----------------------------
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # remove URLs
    text = text.replace("\n", " ").replace("\t", " ")      # remove line breaks
    text = re.sub(r"[^a-z\s]", " ", text)                  # keep only letters/spaces
    text = re.sub(r"\s+", " ", text).strip()               # normalize whitespace
    return text

df["text_clean"] = df["text_raw"].apply(clean_text)

# Drop empty rows
df = df[df["text_clean"].str.len() > 0].copy()
df.reset_index(drop=True, inplace=True)

print("Rows to model:", len(df))

# -----------------------------
# 3) TF–IDF Vectorization
# -----------------------------
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # unigrams + bigrams for interpretability
    min_df=2,             # if too strict, change to 1
    max_df=0.85
)

X = vectorizer.fit_transform(df["text_clean"])
print("TF–IDF matrix shape:", X.shape)

# -----------------------------
# 4) Choose k (4–8) with silhouette (light + defensible)
# -----------------------------
k_candidates = range(4, 9)
sil_scores = {}

for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    sil_scores[k] = score
    print(f"k={k}, silhouette={score:.3f}")

best_k = max(sil_scores, key=sil_scores.get)
print("\nSuggested k (highest silhouette):", best_k)

# Pick k for interpretability (you can override)
k = best_k  # or set k = 5 / 6 manually
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
df["cluster"] = kmeans.fit_predict(X)

# -----------------------------
# 5) Inspect top terms per cluster
# -----------------------------
feature_names = vectorizer.get_feature_names_out()
centroids = kmeans.cluster_centers_

def top_terms(cluster_id: int, top_n: int = 12):
    idx = np.argsort(centroids[cluster_id])[::-1][:top_n]
    return [feature_names[i] for i in idx]

print("\nTop terms per cluster:")
for cid in range(k):
    print(f"\nCluster {cid} (n={sum(df['cluster']==cid)}):")
    print(", ".join(top_terms(cid, top_n=15)))

# -----------------------------
# 6) Show representative examples (so you can name clusters)
# -----------------------------
EXAMPLE_N = 6
for cid in range(k):
    subset = df[df["cluster"] == cid].head(EXAMPLE_N)
    print(f"\n=== Cluster {cid} examples (first {EXAMPLE_N}) ===")
    for _, row in subset.iterrows():
        pid = row.get("primary_id", "")
        sub = row.get("subreddit", "")
        snippet = row["text_raw"].replace("\n", " ")[:240]
        print(f"- {pid} | {sub} | {snippet}...")

# -----------------------------
# 7) Export results for your thematic workflow
# -----------------------------
df.to_csv("reddit_clustered_output.csv", index=False, encoding="utf-8-sig")
print("\nSaved: reddit_clustered_output.csv")


Rows to model: 172
TF–IDF matrix shape: (172, 503)
k=4, silhouette=0.015
k=5, silhouette=0.015
k=6, silhouette=0.015
k=7, silhouette=0.013
k=8, silhouette=0.016

Suggested k (highest silhouette): 8

Top terms per cluster:

Cluster 0 (n=14):
just, headspace, ve, trying, focus, things, adhd, thing, meditation, really, doing, want, need, apps, overwhelmed

Cluster 1 (n=46):
jazz, spotify, playlist, metal, pop, endel, techno, like, soundtrack, death, ambience, album, day, noise, cafe

Cluster 2 (n=22):
music, focus music, focus, favorite, playlist, classical, like, classical music, sleep, good, headspace, sleep music, best, does, sure

Cluster 3 (n=7):
ambient, really, like, soundscapes, waterfall, listening, makes, eno, brian eno, brian, water, running, super, hip hop, hip

Cluster 4 (n=27):
music, focus, deep, work, background, different, love, focused, helps, beats, just, concentrate, need, fi, lo fi

Cluster 5 (n=14):
lofi, listen lofi, youtube, listen, channels, girl, lofi girl, girl 